In [ ]:
import warnings, requests, zipfile, io
warnings.simplefilter('ignore')
import pandas as pd
import boto3

In [ ]:
df = pd.read_csv('s3://自分のバケット名/penguins_binary_classification.csv')

In [ ]:
df.shape

In [ ]:
df.columns

In [ ]:
df.dtypes

In [ ]:
species_mapper = {'Adelie': 0, 'Gentoo': 1}
df['species_encoded'] = df['species'].replace(species_mapper)
df.head(10)

In [ ]:
island_mapper = {'Biscoe': 0, 'Dream': 1, 'Torgersen': 2}
df['island_encoded'] = df['island'].replace(island_mapper)

df.head(10)

In [ ]:
df_for_use = df[['species_encoded',
                 'island_encoded',
                 'bill_length_mm',
                 'bill_depth_mm',
                 'flipper_length_mm',
                 'body_mass_g',
                 'year']].copy()

df_for_use.head()

In [ ]:
!pip install scikit-learn
from sklearn.model_selection import train_test_split

In [ ]:
train, test_and_validate = train_test_split(df_for_use, test_size=0.2, random_state=42, stratify=df['species_encoded'])

In [ ]:
test, validate = train_test_split(test_and_validate, test_size=0.5, random_state=42, stratify=test_and_validate['species_encoded'])

In [ ]:
bucket = '自分のバケット名'
prefix = 'penguin'

train_file    = 'penguin_train.csv'
test_file     = 'penguin_test.csv'
validate_file = 'penguin_validate.csv'

import os

s3_resource = boto3.Session().resource('s3')
def upload_s3_csv(filename, folder, dataframe):
    csv_buffer = io.StringIO()
    dataframe.to_csv(csv_buffer, header=False, index=False)
    s3_resource.Bucket(bucket).Object(os.path.join(prefix, folder, filename)).put(Body=csv_buffer.getvalue())

upload_s3_csv(train_file, 'train', train)

upload_s3_csv(test_file, 'test', test)

upload_s3_csv(validate_file, 'validate', validate)

In [ ]:
import boto3
from sagemaker.image_uris import retrieve

container = retrieve('xgboost',boto3.Session().region_name,'1.0-1')

In [ ]:
hyperparams={"num_round":"42",
             "eval_metric": "auc",
             "objective": "binary:logistic"}

In [ ]:
import sagemaker

s3_output_location="s3://{}/{}/output/".format(bucket,prefix)

xgb_model=sagemaker.estimator.Estimator(container,
                                       sagemaker.get_execution_role(),
                                       instance_count=1,
                                       instance_type='ml.m5.xlarge',
                                       output_path=s3_output_location,
                                        hyperparameters=hyperparams,
                                        sagemaker_session=sagemaker.Session())

In [ ]:
train_channel = sagemaker.inputs.TrainingInput(
    "s3://{}/{}/train/".format(bucket,prefix,train_file),
    content_type='text/csv')

validate_channel = sagemaker.inputs.TrainingInput(
    "s3://{}/{}/validate/".format(bucket,prefix,validate_file),
    content_type='text/csv')

data_channels = {'train': train_channel, 'validation': validate_channel}

In [ ]:
xgb_model.fit(inputs=data_channels, logs=False)

print('ready for hosting!')

In [ ]:
xgb_predictor = xgb_model.deploy(initial_instance_count=1,
                serializer = sagemaker.serializers.CSVSerializer(),
                instance_type='ml.m5.xlarge')
print('Hosting completed!')

In [ ]:
row = test.iloc[0:1,1:]
batch_X_csv_buffer = io.StringIO()
row.to_csv(batch_X_csv_buffer, header=False, index=False)
test_row = batch_X_csv_buffer.getvalue()
print(test_row)
xgb_predictor.predict(test_row)

In [ ]:
try:
    xgb_predictor.delete_endpoint(delete_endpoint_config=True)
    print("Endpoint deleted.")
except Exception as e:
    print(f"Endpoint already deleted or not found: {e}")

batch_X = test.iloc[:,1:]

batch_X_file='batch-in.csv'

upload_s3_csv(batch_X_file, 'batch-in', batch_X)

batch_output = "s3://{}/{}/batch-out/".format(bucket,prefix)

batch_input = "s3://{}/{}/batch-in/{}".format(bucket,prefix,batch_X_file)

xgb_transformer = xgb_model.transformer(instance_count=1,
                                       instance_type='ml.m5.xlarge',
                                       strategy='MultiRecord',
                                       assemble_with='Line',
                                       output_path=batch_output)

xgb_transformer.transform(data=batch_input,
                         data_type='S3Prefix',
                         content_type='text/csv',
                         split_type='Line')

xgb_transformer.wait()

print('Batch transformation completed!')

In [ ]:
s3 = boto3.client('s3')

obj = s3.get_object(Bucket=bucket, Key="{}/batch-out/{}".format(prefix,'batch-in.csv.out'))

target_predicted = pd.read_csv(io.BytesIO(obj['Body'].read()),sep=',',names=['target'])

In [ ]:
def binary_convert(x):
    threshold = 0.65
    if x > threshold:
        return 1
    else:
        return 0

target_predicted_binary = target_predicted['target'].apply(binary_convert)

In [ ]:
target_predicted_binary.head(10)

In [ ]:
test.head(10)